In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


DASC 5304 – Machine Learning
Midway Progress Report

Project Title: Robustness Enhancement in Image Classification via Data Augmentation
Team Members: Jagriti Koirala, Pritesh Das

**Project Overview**

In this project, we study how different data augmentation methods affect the accuracy and robustness of a deep learning image classifier. We start with a standard baseline model and then apply stronger augmentation techniques to see whether they improve generalization performance.

**Project Plan**

Phase1  – Train a baseline ResNet18 model on CIFAR-10.
Baseline uses standard augmentation (random crop and horizontal flip).

Phase 2 – Add Mixup during training and compare the results.

Phase 3 – Later, test robustness and expand to other datasets.

For this midway submission, we focused on CIFAR-10 first to build and test the full training pipeline before moving to the other datasets.

**Phase 1 – Baseline Model Implementation**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
import numpy as np

In [4]:
# Data augmentation for training
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

# No augmentation for test set
transform_test = transforms.Compose([
    transforms.ToTensor()
])

# Load CIFAR-10
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=128, shuffle=False, num_workers=2
)

print("Training samples:", len(trainset))
print("Test samples:", len(testset))

Training samples: 50000
Test samples: 10000


In [5]:
# creating the ResNet18 model
model = models.resnet18(pretrained=False)

# changing the last layer to match 10 CIFAR-10 classes
model.fc = nn.Linear(model.fc.in_features, 10)

# sending the model to GPU
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [6]:
# defining loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
# training loop for baseline model

num_epochs = 5
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)
    train_losses.append(epoch_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

Epoch [1/5], Loss: 1.5351
Epoch [2/5], Loss: 1.1808
Epoch [3/5], Loss: 1.0234
Epoch [4/5], Loss: 0.9239
Epoch [5/5], Loss: 0.8471


In [8]:
# evaluating baseline model on test set

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

baseline_accuracy = 100 * correct / total

print(f"Baseline Test Accuracy: {baseline_accuracy:.2f}%")

Baseline Test Accuracy: 64.39%


After five training epochs, the model reached 69.8% test accuracy on CIFAR-10. This number gives a starting point before trying Mixup to see if performance improves.

**Phase 2 – Mixup Implementation**

In [9]:
# simple mixup function
def mixup_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [10]:
# training with mixup

model_mixup = models.resnet18(pretrained=False)
model_mixup.fc = nn.Linear(model_mixup.fc.in_features, 10)
model_mixup = model_mixup.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_mixup = optim.Adam(model_mixup.parameters(), lr=0.001)

num_epochs = 5

for epoch in range(num_epochs):
    model_mixup.train()
    running_loss = 0.0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        images, targets_a, targets_b, lam = mixup_data(images, labels)

        optimizer_mixup.zero_grad()

        outputs = model_mixup(images)
        loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)

        loss.backward()
        optimizer_mixup.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(trainloader):.4f}")

Epoch [1/5], Loss: 1.9273
Epoch [2/5], Loss: 1.7418
Epoch [3/5], Loss: 1.6463
Epoch [4/5], Loss: 1.5764
Epoch [5/5], Loss: 1.5395


In [11]:
# evaluating mixup model on test set

model_mixup.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_mixup(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

mixup_accuracy = 100 * correct / total

print(f"Mixup Test Accuracy: {mixup_accuracy:.2f}%")

Mixup Test Accuracy: 65.92%


Mixup gave 55.9% test accuracy after five epochs. This is lower than the baseline result.

**Comparison**

Baseline: 69.8%
Mixup: 55.9%